# HMR2.0 (4D-Humans) Inference
**EE-243 Project — HumanWild**

This notebook runs 3D human mesh recovery on test images using HMR2.0, the regressor trained on the HumanWild dataset.


## Install dependencies

In [1]:
# Install detectron2 prebuilt wheel (avoids building from source)
import torch
import sys

CUDA = torch.version.cuda.replace(".", "")
TORCH = torch.__version__.split("+")[0].replace(".", "")
print(f"CUDA: {torch.version.cuda}, Torch: {torch.__version__}")

# Install detectron2 for torch 2.11.0 + CUDA 12.8
!pip install --extra-index-url https://miropsota.github.io/torch_packages_builder \
    detectron2==0.6+fd27788pt2.11.0cu128

# Install detectron2 prebuilt wheel
!pip install -q detectron2 -f https://dl.fbaipublicfiles.com/detectron2/wheels/cu121/torch2.5/index.html

CUDA: 12.8, Torch: 2.11.0+cu128
Looking in indexes: https://pypi.org/simple, https://miropsota.github.io/torch_packages_builder
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 57.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 30.4 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=afcc53deaf56aac450890c9b04bbb7fb25c2b742c4cdb38dd76320c97ae51d68
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
Succe

In [2]:
# Clone 4D-Humans repo
!git clone https://github.com/shubham-goel/4D-Humans
%cd 4D-Humans

Cloning into '4D-Humans'...
remote: Enumerating objects: 307, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 307 (delta 104), reused 80 (delta 76), pack-reused 150 (from 1)
Receiving objects: 100% (307/307), 14.34 MiB | 43.83 MiB/s, done.
Resolving deltas: 100% (142/142), done.
/content/4D-Humans


In [3]:
# Install chumpy first (known issue)
!pip install -q --no-build-isolation git+https://github.com/mattloper/chumpy

# Install the rest
!pip install -q --no-build-isolation -e .[all]

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 37.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 55.0 MB/s eta 0:00:00


## Download SMPL model and upload to Colab


In [4]:
import os
from google.colab import files

# Upload the SMPL model file
print("Please upload basicModel_neutral_lbs_10_207_0_v1.0.0.pkl")
uploaded = files.upload()

# Place it in the expected location
smpl_dir = os.path.expanduser("~/.cache/4DHumans/data/smpl")
os.makedirs(smpl_dir, exist_ok=True)

for fname in uploaded.keys():
    os.rename(fname, os.path.join(smpl_dir, fname))
    print(f"Moved {fname} to {smpl_dir}")

Please upload basicModel_neutral_lbs_10_207_0_v1.0.0.pkl


Saving basicModel_neutral_lbs_10_207_0_v1.0.0.pkl to basicModel_neutral_lbs_10_207_0_v1.0.0.pkl
Moved basicModel_neutral_lbs_10_207_0_v1.0.0.pkl to /root/.cache/4DHumans/data/smpl


## Upload test images

In [5]:
import os
from google.colab import files

# Create input folder
os.makedirs("test_images", exist_ok=True)

print("Upload your test images (jpg/png)")
uploaded = files.upload()

for fname in uploaded.keys():
    os.rename(fname, os.path.join("test_images", fname))

imgs = os.listdir("test_images")
print(f"\nUploaded {len(imgs)} images: {imgs}")

Upload your test images (jpg/png)


Saving 1.jpg to 1.jpg
Saving 2.avif to 2.avif
Saving 3.jpg to 3.jpg
Saving 4.jpg to 4.jpg
Saving 5.jpg to 5.jpg
Saving 6.jpg to 6.jpg
Saving 7.jpg to 7.jpg
Saving 8.jpg to 8.jpg
Saving 9.jpg to 9.jpg
Saving 10.jpg to 10.jpg
Saving 11.jpg to 11.jpg
Saving 12.jpg to 12.jpg
Saving 13.jpg to 13.jpg
Saving 14.jpg to 14.jpg
Saving 15.jpg to 15.jpg
Saving 16.jpg to 16.jpg
Saving 17.jpg to 17.jpg
Saving 18.jpg to 18.jpg
Saving 19.jpg to 19.jpg
Saving 20.jpg to 20.jpg
Saving 21.jpg to 21.jpg
Saving 22.jpg to 22.jpg
Saving 23.jpg to 23.jpg
Saving 24.jpg to 24.jpg
Saving 25.jpg to 25.jpg
Saving 26.jpg to 26.jpg
Saving 27.jpg to 27.jpg
Saving 28.jpg to 28.jpg
Saving 29.jpg to 29.jpg
Saving 30.jpg to 30.jpg
Saving 31.jpg to 31.jpg

Uploaded 31 images: ['16.jpg', '24.jpg', '25.jpg', '23.jpg', '31.jpg', '18.jpg', '7.jpg', '8.jpg', '28.jpg', '13.jpg', '17.jpg', '5.jpg', '14.jpg', '15.jpg', '12.jpg', '26.jpg', '21.jpg', '27.jpg', '9.jpg', '20.jpg', '2.avif', '1.jpg', '30.jpg', '19.jpg', '29.jpg', '4.jp

## Run HMR2.0 inference

In [6]:
import shutil, os
os.makedirs("/content/4D-Humans/data/smpl", exist_ok=True)
shutil.copy(
    "/root/.cache/4DHumans/data/smpl/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl",
    "/root/.cache/4DHumans/data/smpl/SMPL_NEUTRAL.pkl"
)
print("Done")

Done


In [7]:
# Fix weights_only issue directly in the lightning source
import re

filepath = "/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py"

with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "return torch.load(",
    "return torch.load("
).replace(
    "weights_only=weights_only",
    "weights_only=False"
)

with open(filepath, "w") as f:
    f.write(content)

print("Patched successfully")

Patched successfully


In [8]:
!python demo.py \
    --img_folder test_images/ \
    --out_folder results/ \
    --batch_size=1 \
    --side_view \
    --full_frame

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
  [============================================================] 100.0% of 2585.0MB file  
Extracting file: hmr2_data.tar.gz
data/
data/smpl/
data/smpl_mean_params.npz
data/SMPL_to_J19.pkl
logs/
logs/train/
logs/train/multiruns/
logs/train/multiruns/hmr2/
logs/train/multiruns/hmr2/0/
logs/train/multiruns/hmr2/0/checkpoints/
logs/train/multiruns/hmr2/0/checkpoints/epoch=35-step=1000000.ckpt
logs/train/multiruns/hmr2/0/model_config.yaml
logs/train/multiruns/hmr2/0/dataset_config.yaml
Lightning automatically upgraded your loaded checkpoint from v1.8.1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../root/.cache/4DHumans/logs/train/multiruns/h

## View results

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

result_imgs = sorted(Path("results").rglob("*.png"))
print(f"Found {len(result_imgs)} result images")

for img_path in result_imgs:
    img = mpimg.imread(img_path)
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.title(img_path.name)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## Download results

In [11]:
from PIL import Image
from pathlib import Path

for f in Path("results").rglob("*_all.png"):
    img = Image.open(f)
    img = img.resize((img.width // 2, img.height // 2))
    img.save(f, optimize=True, quality=85)
print("Done")

Done


In [12]:
import shutil
from google.colab import files

shutil.make_archive("hmr2_results", "zip", "results")
files.download("hmr2_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>